# Glacier domain setup for SUMMA

This notebook builds the glacier part of a SYMFLUENCE domain for SUMMA, one step at a time. Each step runs one
script in this folder (the same ones `setup_glacier_domain.py` runs), then plots what it wrote. The domain's
settings are in `../domains/<domain>.yaml`.

The upland domain comes from the SYMFLUENCE notebooks: the DEM, elevation-band HRUs, the ERA5 forcing remapped
to the HRUs, and the upland `attributes.nc` and `coldState.nc`, copied into `../data/domain_<Name>`, where every
step below reads and writes. This notebook replaces their glacier cells
(2b section 4, 2c *Glacier Domains*, 4a section 4, 4b from *plot this data against observations* on).

| Step | Script | Writes |
| --- | --- | --- |
| 1 | `glacier_rasters.py` | domain type, RGI id, debris thickness and bed rasters |
| 2 | `glacier_extend.py` | the basin on the bed, and the type and id rasters extended downstream |
| — | SYMFLUENCE 2d, 4a | elevation-band HRUs, remapped forcing, upland attributes and cold state |
| 3 | `glacier_zonal_stats.py` | per-HRU statistics of the glacier rasters, and the HRU raster |
| 4 | `correct_forcing_to_station.py` | forcing corrected toward the on-site stations and stakes |
| 5 | `glacier_cold_state.py` | `attributes_glac.nc` and `coldstate_glac.nc` |
| 6 | `build_cases.py` | the tests and full runs of `cases.yaml` |

In [ ]:
import subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import netCDF4 as nc4

HERE = Path.cwd().parent                      # summa_glacier
sys.path.insert(0, str(HERE))
from domain_config import load_config

DOMAIN = "wolverine"                          # or "gulkana"
config_path = HERE / "domains" / f"{DOMAIN}.yaml"
c = load_config(config_path)
project, name = Path(c["project_dir"]), c["domain"]
glacier_dir = project / "attributes" / "glaciers"
dem_dir = project / "attributes" / "elevation" / "dem"
plots_dir = project / "plots" / "catchment"
plots_dir.mkdir(parents=True, exist_ok=True)

def step(*steps):
    subprocess.run([sys.executable, str(HERE / "setup_glacier_domain.py"), str(config_path), "--steps", *steps], check=True)

def show(array, title, cmap="viridis", label="", save=None, **kw):
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(array, cmap=cmap, **kw)
    fig.colorbar(im, ax=ax, shrink=0.7, label=label)
    ax.set_title(title)
    if save:
        fig.savefig(plots_dir / save, dpi=300, bbox_inches="tight")
    plt.show()

## 1. Glacier rasters

Outlines are RGI 6.0, replaced by the Herreid and Pellicciotti (2020) outline where it has one; their ablation
zones and debris cover give the domain type (1 upland, 2 accumulation, 3 clean ablation, 4 debris ablation).
Debris thickness is Rounce et al. (2021), raised to 4 cm where thinner; the bed is the surface less the Millan
et al. (2022) glacier thickness. It also prints each glacier's lateral moraine width for `latMoraineWidth`.

In [ ]:
step("rasters")
read = lambda path: rasterio.open(path).read(1)
show(read(glacier_dir / f"domain_{name}_domain_type.tif"), "Domain type",
     cmap=ListedColormap(["gray", "cyan", "blue", "orange"]), vmin=0.5, vmax=4.5, save=f"{name}_domain_type.png")
show(read(glacier_dir / f"domain_{name}_rgi_id.tif"), "RGI 6.0 id", cmap="tab20", save=f"{name}_rgi_id.png")
debris = read(glacier_dir / f"domain_{name}_debris_thickness.tif")
show(np.where(debris > 0, debris, np.nan), "Debris thickness", cmap="viridis_r", label="m", save=f"{name}_debris_thickness.png")
show(read(dem_dir / f"domain_{name}_elv.tif"), "Surface elevation", label="m", save=f"{name}_elv.png")
show(read(dem_dir / f"domain_{name}_bedrock_elv.tif"), "Bed elevation", label="m", save=f"{name}_bedrock_elv.png")

## 2. Basin on the bed and glacier-capable cells

The basin is delineated on the bed, since a convex glacier surface misplaces the divide. Each glacier at least
80% inside it then gets room to advance: its terminus is dragged down the flow direction until it has added
(glacier cells)^(2/3) cells, or meets the basin edge or another glacier. Needs pysheds (with numpy 2, the fork
`pip install git+https://github.com/ashleymedin/pysheds.git`).

In [ ]:
step("extend")
gid = read(glacier_dir / f"domain_{name}_rgi_id_extend.tif")
dtype = read(glacier_dir / f"domain_{name}_domain_type_extend.tif")
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(dtype, cmap=ListedColormap(["gray", "cyan", "blue", "orange"]), vmin=0.5, vmax=4.5)
ax.contour(gid > 0, levels=[0.5], colors="red", linewidths=1)
ax.set_title("Domain type, with the glacier-capable area outlined")
fig.savefig(plots_dir / f"{name}_catchment_domain_type_rgi_id_extend.png", dpi=300, bbox_inches="tight")
plt.show()

## Upland domain, from SYMFLUENCE

SYMFLUENCE's 2d (elevation bands) and 4a/4b (forcing remapping, upland `attributes.nc` and `coldState.nc`) make the
upland domain; copy it into `../data/domain_<Name>`. The glacier steps read `shapefiles/catchment/<name>_HRUs_elevation.shp`,
`forcing/SUMMA_input` and `settings/SUMMA/{attributes,coldState}.nc` there.

## 3. Glacier statistics per HRU

For each HRU: the mean debris thickness of its debris cells, the cell count of each domain type, and the mean
elevation, tan slope and aspect of each domain type; and the HRU of every DEM cell.

In [ ]:
step("zonal")
import geopandas as gpd
inter = project / "shapefiles" / "catchment_intersection"
stats = gpd.read_file(inter / "with_dem_domain" / "catchment_with_dem_domain.shp").drop(columns="geometry")
counts = gpd.read_file(inter / "with_domain_type" / "catchment_with_domain_type.shp").filter(like="domType")
debris = gpd.read_file(inter / "with_debris_thickness" / "catchment_with_debris.shp")[["debri_mean"]]
pd.concat([stats.filter(regex="HRU_ID|elv_mean|tsl_mean|asp_mean"), counts, debris], axis=1)

## 4. Station-corrected forcing

`correct_forcing_to_station.py` moves the ERA5 forcing toward the USGS benchmark-glacier stations: air
temperature by an offset per HRU, calendar month and wet or dry hour, with the station lapsed to the HRU by
that month's two-station lapse rate; humidity by a ratio on the same split; shortwave and pressure by one ratio
per HRU; wind by the monthly station-to-forcing ratio (SUMMA's `windExposure` then scales it to the surface).
Precipitation is scaled to the stakes' winter balances, snowfall only and inside the stake intervals, and with a
gauge shield HRUs without stakes take the undercatch-corrected gauge. Gulkana's stations have no humidity over
the forcing years, so its April–August ratios come from Wilson et al.'s 2024 Site B station bridged through
ARCO-ERA5 (`spechum_ratio_from_campaign.py`, needs gcsfs and zarr). Longwave is unchanged.

In [ ]:
step("forcing")
corrections = pd.read_csv(project / "forcing" / "SUMMA_input" / "station_corrected" / "corrections.csv")
corrections.head(24)

## 5. Glacier attributes and cold state

Each HRU gets an upland domain, glacier domains where it has glacier-capable cells, and a debris domain with a
clean one beside it wherever the glacier has debris. Clean ice has five layers (0.15, 0.30, 1.80, 4.75, 23 m).
A debris domain puts 2 to 5 soil layers over the same ice, enough for the thickest debris cell in its HRU: the
top layer at most 5 cm, each layer below a fixed ratio times the one above, as SUMMA re-splits the debris at
each glacier update. Ice starts at −5 °C with ice fraction 0.90 to 0.98 down the column; soil and debris start
1.5 °C above the HRU's mean air temperature over the first 12 forcing months (`cold_start` in the YAML), the
canopy at that air temperature. Each test or run writes its own cold state from its own first month (step 6).

In [ ]:
step("coldstate")
cs = nc4.Dataset(project / "settings" / "SUMMA" / "coldstate_glac.nc")
rows = []
for h, hru in enumerate(cs["hruId"][:]):
    for k in range(cs.dimensions["dom"].size):
        t = int(cs["domType"][0, h, k])
        if t == 0:
            continue
        n = [int(cs[v][0, h, k]) for v in ("nSnow", "nLake", "nSoil", "nGlce")]
        top = sum(n[:3])
        rows.append({"hru": int(hru), "domType": t, "area_km2": cs["DOMarea"][0, h, k] / 1e6, "elev_m": cs["DOMelev"][0, h, k],
                     "nSoil": n[2], "nGlce": n[3], "soil_depths_m": np.round(cs["mLayerDepth"][n[0] + n[1]:top, h, k], 3).tolist(),
                     "soil_T_degC": np.round(cs["mLayerTemp"][n[0] + n[1], h, k] - 273.16, 2) if n[2] else None})
cs.close()
pd.DataFrame(rows)

## 6. Tests and runs

`build_cases.py` builds every case of `cases.yaml` from the domains: the short Gulkana and Wolverine tests,
summa's two-GRU `gulkana_wolverine` test, and the four full 2008–2022 runs (raw and station-corrected forcing).
Each gets the domain's attributes and a cold state started from its own first forcing month; Wolverine's cases
get a stream HRU and a one-reach topology (summa's `add_stream_hru.py`).

In [ ]:
subprocess.run([sys.executable, str(HERE / "build_cases.py"), str(HERE / "cases.yaml")], check=True)